# Conquer Mobile — GitHub Download + 3D Structural Audit

تشغيل خلية واحدة فقط: تسحب حزمة الأصول مباشرة من GitHub Release المحدد داخل الكود، تتحقق من الحجم وSHA-256، تفك الضغط بأمان، تفحص ملفات 3D، ثم ترفع تقارير JSON/CSV/HTML إلى نفس الإصدار.

**لا يوجد Google Drive أو اختيار ملفات أو طلب مسار/رابط من المستخدم.** تحتاج فقط إلى Secret باسم `GITHUB_TOKEN` مع صلاحية رفع أصول GitHub، وفعّل Notebook access.

In [ ]:
%pip -q install requests trimesh

import os, json, csv, html, gc, hashlib, shutil, stat, zipfile, requests
from pathlib import Path, PurePosixPath
from google.colab import userdata
import trimesh

# Fixed source configuration: the notebook fetches the archive directly from GitHub.
# No Google Drive, no folder path prompt, no file picker, and no manual URL input.
OWNER = "ZzwahwahzZ"
REPO = "Conquer-Mobile-Assets."
TAG = "mobile-extracted-assets"
BUNDLE_NAME = "extracted-assets-001.zip"
WORK = Path("/content/conquer_mobile_github_pipeline")
ARCHIVE = WORK / BUNDLE_NAME
EXTRACTED = WORK / "extracted"
REPORTS = WORK / "3d_model_audit"
API = f"https://api.github.com/repos/{OWNER}/{REPO}"
MODEL_EXTS = {".glb", ".gltf", ".obj", ".fbx", ".dae", ".blend", ".3ds", ".stl", ".ply"}
CHUNK = 8 * 1024 * 1024
SAFETY_BYTES = 512 * 1024 * 1024

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(CHUNK), b""):
            h.update(block)
    return h.hexdigest()

def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    count = total = 0
    with zipfile.ZipFile(zip_path) as archive:
        bad = archive.testzip()
        if bad:
            raise RuntimeError(f"ZIP CRC test failed at: {bad}")
        for info in archive.infolist():
            raw = info.filename.replace("\\", "/")
            member = PurePosixPath(raw)
            if member.is_absolute() or ".." in member.parts or not member.parts:
                raise RuntimeError(f"Unsafe ZIP path rejected: {info.filename}")
            mode = (info.external_attr >> 16) & 0xFFFF
            if stat.S_ISLNK(mode):
                raise RuntimeError(f"Symlink rejected: {info.filename}")
            target = (destination / Path(*member.parts)).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(f"ZIP path escapes destination: {info.filename}")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info, "r") as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, length=CHUNK)
            count += 1
            total += info.file_size
    return count, total

def github_headers(token):
    return {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

def upload_report(session, release, path):
    upload_url = release["upload_url"].split("{", 1)[0]
    assets_url = f'{API}/releases/{release["id"]}/assets'
    assets_response = session.get(assets_url, timeout=60)
    assets_response.raise_for_status()
    existing = next((a for a in assets_response.json() if a["name"] == path.name), None)
    if existing:
        deleted = session.delete(existing["url"], timeout=60)
        deleted.raise_for_status()
    with open(path, "rb") as stream:
        response = session.post(
            upload_url,
            params={"name": path.name},
            headers={**session.headers, "Content-Type": "application/octet-stream"},
            data=stream,
            timeout=(60, 900),
        )
    response.raise_for_status()
    asset = response.json()
    if asset.get("name") != path.name or asset.get("size") != path.stat().st_size:
        raise RuntimeError(f"Upload verification failed for {path.name}")
    print(f"UPLOADED + VERIFIED: {asset['name']} ({asset['size']:,} bytes)")

print("Preparing workspace...")
WORK.mkdir(parents=True, exist_ok=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
if EXTRACTED.exists():
    shutil.rmtree(EXTRACTED)
EXTRACTED.mkdir(parents=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print("Reading the fixed GitHub Release source...")
release_response = requests.get(f"{API}/releases/tags/{TAG}", timeout=60)
release_response.raise_for_status()
release = release_response.json()
asset = next((a for a in release.get("assets", []) if a["name"] == BUNDLE_NAME), None)
if asset is None:
    raise FileNotFoundError(f"{BUNDLE_NAME} was not found in GitHub release {TAG}")
expected_size = int(asset["size"])
free = shutil.disk_usage("/content").free
print(f"GitHub source: {release['html_url']}")
print(f"Archive: {BUNDLE_NAME} | {expected_size:,} bytes")
print(f"Colab free disk: {free:,} bytes")
if free < expected_size + SAFETY_BYTES:
    raise RuntimeError("Insufficient Colab disk space for download and extraction. No extraction started.")

print("Downloading the archive directly from GitHub...")
download_url = asset["browser_download_url"]
h = hashlib.sha256()
downloaded = 0
with requests.get(download_url, stream=True, timeout=(30, 300)) as response:
    response.raise_for_status()
    with open(ARCHIVE, "wb") as out:
        for block in response.iter_content(chunk_size=CHUNK):
            if block:
                out.write(block)
                h.update(block)
                downloaded += len(block)
if downloaded != expected_size:
    raise RuntimeError(f"Download size mismatch: expected {expected_size:,}, got {downloaded:,}")
archive_sha256 = h.hexdigest()
print(f"DOWNLOAD VERIFIED: {downloaded:,} bytes | SHA-256 {archive_sha256}")

print("Checking uncompressed size and available disk...")
with zipfile.ZipFile(ARCHIVE) as archive:
    bad = archive.testzip()
    if bad:
        raise RuntimeError(f"ZIP CRC test failed at: {bad}")
    uncompressed_bytes = sum(i.file_size for i in archive.infolist() if not i.is_dir())
free = shutil.disk_usage("/content").free
print(f"Uncompressed bytes: {uncompressed_bytes:,} | Free disk: {free:,}")
if free < uncompressed_bytes + SAFETY_BYTES:
    raise RuntimeError("Insufficient free disk space for safe extraction. No extraction started.")

print("Safely extracting GitHub archive...")
extracted_count, extracted_bytes = safe_extract(ARCHIVE, EXTRACTED)
if extracted_count == 0:
    raise RuntimeError("The GitHub archive extracted zero files; refusing to continue.")
print(f"EXTRACTED: {extracted_count:,} files | {extracted_bytes:,} bytes")

model_paths = sorted(
    (p for p in EXTRACTED.rglob("*") if p.is_file() and p.suffix.lower() in MODEL_EXTS),
    key=lambda p: p.relative_to(EXTRACTED).as_posix().casefold()
)
print(f"Starting structural audit of {len(model_paths):,} 3D files...")
audit_rows = []
for index, path in enumerate(model_paths, 1):
    rel = path.relative_to(EXTRACTED).as_posix()
    row = {
        "path": rel, "extension": path.suffix.lower(),
        "size_bytes": path.stat().st_size, "sha256": None,
        "status": "not_tested", "geometry_count": 0,
        "vertex_count": 0, "face_count": 0,
        "bounds": "", "extents": "", "watertight_meshes": 0, "error": ""
    }
    loaded = None
    try:
        row["sha256"] = sha256_file(path)
        loaded = trimesh.load(str(path), force="scene", process=False)
        if isinstance(loaded, trimesh.Trimesh):
            geometries = [loaded]
        elif isinstance(loaded, trimesh.Scene):
            geometries = list(loaded.geometry.values())
        else:
            geometries = []
        meshes = [g for g in geometries if isinstance(g, trimesh.Trimesh)]
        row["geometry_count"] = len(geometries)
        row["vertex_count"] = sum(len(m.vertices) for m in meshes)
        row["face_count"] = sum(len(m.faces) for m in meshes)
        row["watertight_meshes"] = sum(1 for m in meshes if m.is_watertight)
        bounds = loaded.bounds if hasattr(loaded, "bounds") else None
        if bounds is not None:
            row["bounds"] = json.dumps(bounds.tolist())
            row["extents"] = json.dumps((bounds[1] - bounds[0]).tolist())
        if not meshes:
            row["status"] = "parsed_no_triangle_mesh"
        elif row["vertex_count"] == 0 or row["face_count"] == 0:
            row["status"] = "empty_or_degenerate_mesh"
        else:
            row["status"] = "parsed_mesh"
    except Exception as exc:
        row["status"] = "parser_error_or_unsupported"
        row["error"] = f"{type(exc).__name__}: {str(exc)[:500]}"
    audit_rows.append(row)
    if index % 10 == 0 or index == len(model_paths):
        print(f"Audited {index}/{len(model_paths)}: {rel} -> {row['status']}")
    if loaded is not None:
        del loaded
    gc.collect()

counts = {}
for row in audit_rows:
    counts[row["status"]] = counts.get(row["status"], 0) + 1

json_path = REPORTS / "3d-model-audit.json"
csv_path = REPORTS / "3d-model-audit.csv"
html_path = REPORTS / "3d-model-audit.html"
summary = {
    "source_release": release["html_url"],
    "source_asset": BUNDLE_NAME,
    "source_size_bytes": downloaded,
    "source_sha256": archive_sha256,
    "extracted_files": extracted_count,
    "extracted_bytes": extracted_bytes,
    "total_models": len(audit_rows),
    "status_counts": counts,
    "note": (
        "Structural parser audit only. Parser errors may indicate unsupported formats "
        "or missing dependencies and do not by themselves prove corruption. Visual appearance, "
        "rigging, animation, textures, and Godot compatibility require separate checks."
    ),
    "files": audit_rows
}
json_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
fields = list(audit_rows[0].keys()) if audit_rows else [
    "path","extension","size_bytes","sha256","status","geometry_count",
    "vertex_count","face_count","bounds","extents","watertight_meshes","error"
]
with open(csv_path, "w", newline="", encoding="utf-8-sig") as stream:
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    writer.writerows(audit_rows)

table_rows = "".join(
    "<tr>" + "".join(f"<td>{html.escape(str(row.get(k, '')))}</td>" for k in
    ["path","extension","size_bytes","status","geometry_count","vertex_count","face_count","error"]) + "</tr>"
    for row in audit_rows
)
html_doc = f"""<!doctype html><html><head><meta charset="utf-8">
<title>3D Model Structural Audit</title>
<style>body{{font-family:Arial,sans-serif;margin:24px}}table{{border-collapse:collapse;width:100%;font-size:13px}}
th,td{{border:1px solid #ddd;padding:7px;text-align:left;vertical-align:top}}
th{{position:sticky;top:0;background:#eee}}tr:nth-child(even){{background:#f7f7f7}}</style>
</head><body><h1>3D Model Structural Audit</h1>
<p>Models: {len(audit_rows)} | Extracted files: {extracted_count}</p>
<pre>{html.escape(json.dumps(counts,ensure_ascii=False,indent=2))}</pre>
<p>{html.escape(summary["note"])}</p>
<table><thead><tr><th>Path</th><th>Format</th><th>Bytes</th><th>Status</th>
<th>Geometries</th><th>Vertices</th><th>Faces</th><th>Error</th></tr></thead>
<tbody>{table_rows}</tbody></table></body></html>"""
html_path.write_text(html_doc, encoding="utf-8")

print("\nAUDIT SUMMARY")
print(json.dumps(counts, ensure_ascii=False, indent=2))
print("Created:", json_path.name, csv_path.name, html_path.name)

print("Connecting to GitHub for report publishing...")
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN was not found in Colab Secrets. Add your existing GitHub token "
        "as GITHUB_TOKEN and enable Notebook access."
    )
session = requests.Session()
session.headers.update(github_headers(token))
who = session.get("https://api.github.com/user", timeout=60)
who.raise_for_status()
print("Authenticated to GitHub as:", who.json().get("login"))
repo_check = session.get(API, timeout=60)
repo_check.raise_for_status()
auth_release_response = session.get(f"{API}/releases/tags/{TAG}", timeout=60)
auth_release_response.raise_for_status()
auth_release = auth_release_response.json()

for report_path in [json_path, csv_path, html_path]:
    upload_report(session, auth_release, report_path)

verify = session.get(f'{API}/releases/{auth_release["id"]}/assets', timeout=60)
verify.raise_for_status()
remote = {a["name"]: a for a in verify.json()}
for path in [json_path, csv_path, html_path]:
    item = remote.get(path.name)
    if not item or item["size"] != path.stat().st_size:
        raise RuntimeError(f"Final GitHub release verification failed for {path.name}")

print("\nSUCCESS: GitHub archive downloaded, size/hash verified, safely extracted,")
print("3D models structurally audited, and all three reports uploaded and verified.")
print(f"Release: https://github.com/{OWNER}/{REPO}/releases/tag/{TAG}")
print(f"3D models audited: {len(audit_rows)}")
